# Building the Graph [Step 3 - NetworkX In Memory, Neo4j in Production]

> **MLCourse - Agentic AI - Advanced RAG - Graph RAG**

We have triples. This notebook turns them into an actual graph object, inspects
its structure, and - crucially - learns to spot the two structural defects that
silently break Graph RAG: **disconnected components** and **isolated nodes**.

We use **NetworkX**: a pure-Python, in-memory graph library. No database, no
server, no Docker. It is genuinely sufficient to tens of thousands of nodes,
which covers most single-corpus knowledge graphs. The production alternative,
Neo4j, is covered in section 7 - shown, explained, and **not required**.

### 1. Setup


In [1]:
import os
import re
import time
import json
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
from dotenv import load_dotenv


def find_env(start=None):
    """Walk up from the notebook directory until a .env file appears."""
    start = Path(start or Path.cwd()).resolve()
    for folder in [start, *start.parents]:
        candidate = folder / ".env"
        if candidate.exists():
            return candidate
    raise FileNotFoundError("No .env found walking up from " + str(start))


ENV_PATH = find_env()
load_dotenv(ENV_PATH)
DATA_DIR = ENV_PATH.parent / "data"

print("env file :", ENV_PATH)
print("data dir :", DATA_DIR)
print("GROQ_API_KEY present:", bool(os.environ.get("GROQ_API_KEY")))

env file : D:\projects\python\MLCourse\03_agentic_ai\.env
data dir : D:\projects\python\MLCourse\03_agentic_ai\data
GROQ_API_KEY present: True


In [2]:
from langchain_groq import ChatGroq

GROQ_MODEL = "qwen/qwen3.8-27b"          # verified available on this account
llm = ChatGroq(model=GROQ_MODEL, temperature=0)

THINK_RE = re.compile(r"<think>.*?</think>", re.DOTALL)


def clean(text):
    """Strip any <think>...</think> block a reasoning model may emit."""
    return THINK_RE.sub("", text).strip()


def ask(prompt, retries=4, pause=1.5):
    """Call Groq with exponential backoff. Free tier is roughly 8000 tokens/minute,
    so every loop in these notebooks paces itself and retries on rate limits."""
    delay = 5.0
    for attempt in range(retries):
        try:
            answer = clean(llm.invoke(prompt).content)
            time.sleep(pause)
            return answer
        except Exception as exc:
            if attempt == retries - 1:
                raise
            print(f"  [retry {attempt + 1}] {type(exc).__name__} - sleeping {delay:.0f}s")
            time.sleep(delay)
            delay *= 2


print("Groq model:", GROQ_MODEL)
print("smoke test:", ask("Reply with exactly one word: ready"))

Groq model: qwen/qwen3.8-27b


smoke test: ready


In [3]:
ALICE_PATH = DATA_DIR / "alice.txt"
raw_text = ALICE_PATH.read_text(encoding="utf-8-sig")

paragraphs = [" ".join(p.split()) for p in raw_text.split("\n\n") if len(p.strip()) > 200]
print("paragraphs:", len(paragraphs))

paragraphs: 237


### Extracted with the LLM in notebook 02 and saved here so notebooks 03-05 do not


In [ ]:
# each re-pay the extraction cost. In a real system this is exactly what you do:
# extraction is a ONE-OFF indexing step whose output is persisted.
TRIPLES = [
    ("Alice", "follows", "White Rabbit"),
    ("Alice", "falls down", "rabbit hole"),
    ("White Rabbit", "carries", "pocket watch"),
    ("White Rabbit", "serves", "Duchess"),
    ("White Rabbit", "is herald for", "King of Hearts"),
    ("Alice", "drinks from", "little bottle"),
    ("little bottle", "causes", "shrinking"),
    ("Alice", "eats", "cake"),
    ("cake", "causes", "growing"),
    ("Alice", "meets", "Caterpillar"),
    ("Caterpillar", "sits on", "mushroom"),
    ("mushroom", "causes", "size change"),
    ("Caterpillar", "advises", "Alice"),
    ("Alice", "meets", "Cheshire Cat"),
    ("Cheshire Cat", "belongs to", "Duchess"),
    ("Cheshire Cat", "vanishes leaving", "grin"),
    ("Cheshire Cat", "directs Alice to", "Mad Hatter"),
    ("Alice", "attends", "mad tea party"),
    ("Mad Hatter", "attends", "mad tea party"),
    ("March Hare", "attends", "mad tea party"),
    ("Dormouse", "attends", "mad tea party"),
    ("Mad Hatter", "quarrelled with", "Time"),
    ("Duchess", "nurses", "baby"),
    ("baby", "turns into", "pig"),
    ("Duchess", "employs", "Cook"),
    ("Cook", "throws", "pepper"),
    ("Alice", "meets", "Queen of Hearts"),
    ("Queen of Hearts", "orders", "beheadings"),
    ("Queen of Hearts", "plays", "croquet"),
    ("croquet", "uses", "flamingo"),
    ("croquet", "uses", "hedgehog"),
    ("Queen of Hearts", "is married to", "King of Hearts"),
    ("Queen of Hearts", "commands", "playing cards"),
    ("playing cards", "paint", "white roses"),
    ("Knave of Hearts", "is accused of stealing", "tarts"),
    ("Queen of Hearts", "baked", "tarts"),
    ("King of Hearts", "presides over", "trial"),
    ("Knave of Hearts", "stands at", "trial"),
    ("Mad Hatter", "testifies at", "trial"),
    ("Alice", "testifies at", "trial"),
    ("Gryphon", "takes Alice to", "Mock Turtle"),
    ("Queen of Hearts", "sends", "Gryphon"),
    ("Mock Turtle", "tells", "his history"),
    ("Mock Turtle", "dances", "Lobster Quadrille"),
    ("Gryphon", "dances", "Lobster Quadrille"),
]

print(len(TRIPLES), "curated (subject, relation, object) triples")


### 2. Directed, labelled, multi-edge: choosing the graph type

Three modelling decisions, each with a consequence:

- **Directed** (`DiGraph`). Relations have direction: *Queen orders beheadings*
  is not *beheadings order Queen*. Use a directed graph, then decide per query
  whether to traverse edges in both directions.
- **Labelled edges.** The relation name lives on the edge as an attribute. This
  is what distinguishes a knowledge graph from a plain network.
- **Multi-edge** (`MultiDiGraph`). Two entities can be connected by more than one
  relation - Alice both *meets* and *testifies with* the Hatter. A plain
  `DiGraph` silently overwrites the first edge with the second.

We use `MultiDiGraph` for exactly that reason.

In [5]:
import networkx as nx

G = nx.MultiDiGraph()

for subject, relation, obj in TRIPLES:
    G.add_node(subject)
    G.add_node(obj)
    G.add_edge(subject, obj, relation=relation)

print("nodes:", G.number_of_nodes())
print("edges:", G.number_of_edges())
print("distinct relation types:",
      len({d["relation"] for _, _, d in G.edges(data=True)}))

nodes: 38
edges: 45
distinct relation types: 35


### What a single node looks like from the inside.


In [ ]:
node = "Cheshire Cat"
print(f"--- {node} ---")
print("outgoing:")
for _, target, data in G.out_edges(node, data=True):
    print(f"   --[{data['relation']}]--> {target}")
print("incoming:")
for source, _, data in G.in_edges(node, data=True):
    print(f"   {source} --[{data['relation']}]-->")


### 3. Reading the structure

Before querying a graph, look at its shape. Three cheap diagnostics tell you
whether it is usable.

**Degree** identifies hubs - the entities most facts attach to. In a healthy
graph these are the ones you would expect a domain expert to name.

In [7]:
degrees = sorted(G.degree(), key=lambda kv: -kv[1])

print(f"{'entity':<24}{'degree':>8}{'out':>6}{'in':>5}")
print("-" * 43)
for name, deg in degrees[:12]:
    print(f"{name:<24}{deg:>8}{G.out_degree(name):>6}{G.in_degree(name):>5}")

entity                    degree   out   in
-------------------------------------------
Alice                         10     9    1
Queen of Hearts                7     6    1
White Rabbit                   4     3    1
Duchess                        4     2    2
Cheshire Cat                   4     3    1
Mad Hatter                     4     3    1
mad tea party                  4     0    4
trial                          4     0    4
King of Hearts                 3     1    2
Caterpillar                    3     2    1
croquet                        3     2    1
Gryphon                        3     2    1


### Relation vocabulary - a long tail here means schema drift crept in.


In [ ]:
from collections import Counter

rel_counts = Counter(d["relation"] for _, _, d in G.edges(data=True))
print(f"{'relation':<26}{'count':>7}")
print("-" * 33)
for rel, n in rel_counts.most_common():
    print(f"{rel:<26}{n:>7}")


### 4. The two structural defects that break Graph RAG

**Disconnected components.** If your graph is really three separate graphs, no
traversal can ever get from one to another - and the failure is silent. It
returns an empty result, not an error. Almost always this means entity
resolution failed: "Queen" and "Queen of Hearts" became two nodes.

**Isolated nodes.** Degree-1 leaves are fine; genuinely disconnected nodes are
extraction noise that adds size without adding answerable structure.

In [9]:
undirected = G.to_undirected()
components = sorted(nx.connected_components(undirected), key=len, reverse=True)

print(f"connected components: {len(components)}")
for i, comp in enumerate(components, 1):
    print(f"  component {i}: {len(comp)} nodes "
          f"({'main' if i == 1 else 'ORPHANED - check entity resolution'})")
    if i > 1 or len(comp) < 12:
        print("    ", sorted(comp))

coverage = len(components[0]) / G.number_of_nodes()
print(f"\nlargest component covers {coverage:.0%} of nodes "
      f"- above ~90% is healthy")

connected components: 1
  component 1: 38 nodes (main)

largest component covers 100% of nodes - above ~90% is healthy


In [10]:
leaves = [n for n, d in G.degree() if d == 1]
print(f"degree-1 nodes: {len(leaves)} of {G.number_of_nodes()}")
print("  ", sorted(leaves)[:12])
print("\nLeaves are normal (they are facts about a hub). Whole disconnected "
      "components are not.")

degree-1 nodes: 16 of 38
   ['Dormouse', 'March Hare', 'Time', 'beheadings', 'flamingo', 'grin', 'growing', 'hedgehog', 'his history', 'pepper', 'pig', 'pocket watch']

Leaves are normal (they are facts about a hub). Whole disconnected components are not.


### 5. Paths: the thing a graph can do that an index cannot

This is the payoff. A path query answers *"how is A related to B"* - a question
with no vector-search equivalent at all.

In [11]:
def show_path(graph, source, target):
    try:
        nodes = nx.shortest_path(graph.to_undirected(as_view=False), source, target)
    except (nx.NetworkXNoPath, nx.NodeNotFound) as exc:
        print(f"no path {source} -> {target} ({type(exc).__name__})")
        return None
    parts = []
    for a, b in zip(nodes, nodes[1:]):
        rels = [d["relation"] for _, _, d in G.out_edges(a, data=True) if _ == a and b in G[a]]
        rel = next((d["relation"] for x, y, d in G.edges(data=True)
                    if {x, y} == {a, b}), "related to")
        parts.append(f"{a} --[{rel}]--> ")
    print("  " + "".join(parts) + nodes[-1] + f"   ({len(nodes) - 1} hops)")
    return nodes


print("How is the Queen of Hearts connected to the Mock Turtle?")
show_path(G, "Queen of Hearts", "Mock Turtle")

print("\nHow is Alice connected to the Duchess?")
show_path(G, "Alice", "Duchess")

print("\nHow is the little bottle connected to the trial?")
show_path(G, "little bottle", "trial")

How is the Queen of Hearts connected to the Mock Turtle?
  Queen of Hearts --[sends]--> Gryphon --[takes Alice to]--> Mock Turtle   (2 hops)

How is Alice connected to the Duchess?
  Alice --[follows]--> White Rabbit --[serves]--> Duchess   (2 hops)

How is the little bottle connected to the trial?
  little bottle --[drinks from]--> Alice --[testifies at]--> trial   (2 hops)


['little bottle', 'Alice', 'trial']

Each of those chains is an answer assembled from facts stated in different
paragraphs. No single chunk contains any of them.

### 6. Neighbourhoods and centrality

Two more graph operations that map directly onto retrieval needs:

- **k-hop neighbourhood** - "everything within 2 steps of Alice" is a
  context-gathering operation (notebook 04 builds retrieval on it).
- **Betweenness centrality** - which node lies on the most shortest paths. These
  are the *connectors*: remove them and the graph fragments.

In [12]:
def neighbourhood(graph, center, hops=2):
    return nx.ego_graph(graph.to_undirected(as_view=False), center, radius=hops)


for hops in [1, 2, 3]:
    ego = neighbourhood(G, "Alice", hops)
    print(f"within {hops} hop(s) of Alice: {ego.number_of_nodes()} nodes")

print("\n1-hop neighbours of Alice:", sorted(neighbourhood(G, "Alice", 1).nodes()))

within 1 hop(s) of Alice: 10 nodes
within 2 hop(s) of Alice: 26 nodes
within 3 hop(s) of Alice: 35 nodes

1-hop neighbours of Alice: ['Alice', 'Caterpillar', 'Cheshire Cat', 'Queen of Hearts', 'White Rabbit', 'cake', 'little bottle', 'mad tea party', 'rabbit hole', 'trial']


In [13]:
central = sorted(nx.betweenness_centrality(G.to_undirected(as_view=False)).items(),
                 key=lambda kv: -kv[1])
print(f"{'entity':<24}{'betweenness':>13}")
print("-" * 37)
for name, score in central[:8]:
    print(f"{name:<24}{score:>13.3f}")
print("\nHigh-betweenness nodes are the bridges between regions of the story - "
      "exactly the answer to 'what connects X to Y' questions.")

entity                    betweenness
-------------------------------------
Alice                           0.642
Queen of Hearts                 0.497
Duchess                         0.209
White Rabbit                    0.177
Cheshire Cat                    0.175
Gryphon                         0.153
mad tea party                   0.125
croquet                         0.107

High-betweenness nodes are the bridges between regions of the story - exactly the answer to 'what connects X to Y' questions.


### 7. Persisting it

A NetworkX graph is an in-memory Python object: it disappears with the kernel.
For anything reusable, serialise it. JSON keeps it human-readable and diffable,
which matters when you are debugging extraction.

In [14]:
GRAPH_PATH = Path.cwd() / "alice_graph.json"

payload = {
    "nodes": sorted(G.nodes()),
    "edges": [{"source": u, "relation": d["relation"], "target": v}
              for u, v, d in G.edges(data=True)],
}
GRAPH_PATH.write_text(json.dumps(payload, indent=1), encoding="utf-8")

reloaded = json.loads(GRAPH_PATH.read_text(encoding="utf-8"))
H = nx.MultiDiGraph()
for e in reloaded["edges"]:
    H.add_edge(e["source"], e["target"], relation=e["relation"])

print("saved to :", GRAPH_PATH.name, f"({GRAPH_PATH.stat().st_size} bytes)")
print("round-trip:", H.number_of_nodes(), "nodes,", H.number_of_edges(), "edges")
print("identical :", H.number_of_edges() == G.number_of_edges())

saved to : alice_graph.json (4831 bytes)
round-trip: 38 nodes, 45 edges
identical : True


### 8. Neo4j - the production option (not required here)

NetworkX runs out of road when you need persistence across processes, concurrent
writers, indexed lookups over millions of nodes, or traversals that will not fit
in RAM. That is when you move to a graph database, and **Neo4j** is the usual
choice.

The same questions become **Cypher**:

```cypher
// who owns the cat that directed Alice to the Hatter? (the 2-hop question)
MATCH (cat)-[:DIRECTS_ALICE_TO]->(:Character {name: 'Mad Hatter'}),
      (cat)-[:BELONGS_TO]->(owner)
RETURN owner.name

// who is at both the tea party and the trial? (the aggregation question)
MATCH (c:Character)-[:ATTENDS]->(:Event {name: 'mad tea party'}),
      (c)-[:TESTIFIES_AT]->(:Event {name: 'trial'})
RETURN c.name
```

And LangChain has first-class support - `Neo4jGraph` plus `GraphCypherQAChain`
will even write the Cypher from a natural-language question.

**Running it needs a server**, typically:

```bash
docker run -p 7474:7474 -p 7687:7687 -e NEO4J_AUTH=neo4j/password neo4j:5
```

**This course does not require Docker.** Everything in this module runs
in-process on NetworkX, and every notebook here executes without any external
service. Treat the Cypher above as the thing you graduate to when the in-memory
graph stops fitting - not as a prerequisite for the rest of the module.

Rough guidance on when to switch:

| | NetworkX | Neo4j |
|---|---|---|
| scale | up to ~10^4-10^5 nodes | 10^6+ |
| persistence | serialise it yourself | built in |
| concurrent access | no | yes |
| query language | Python | Cypher |
| operational cost | none | a server to run |

### 9. Key takeaways

- Use a **`MultiDiGraph`**: directed because relations have direction,
  multi-edge because two entities can be related in several ways.
- Check **connected components** first. More than one usually means entity
  resolution failed, and traversals will silently return nothing.
- **Paths** and **k-hop neighbourhoods** are the operations that have no
  vector-search equivalent.
- **Betweenness centrality** finds the connectors - the answer to "what links X
  and Y".
- Serialise the graph; NetworkX holds nothing across a restart.

Next: [`04_graph_traversal_retrieval.ipynb`](04_graph_traversal_retrieval.ipynb)
- using these operations as a retrieval system.